In [0]:
run_date = dbutils.widgets.get("run_date")
print("Run model fitting and prediction for {}".format(run_date))

In [0]:
db = 'revr_bmbs_dmp_offline'
table_names = {'mmc_output': 'recommender_output_mmc',
               'lpo_output': 'recommender_output_lpo',
               'combined_output': 'recommender_output'
               }

In [0]:
def dynamic_write_in_batch_mode(df, table_name, is_partitioned=False, partition_col=None, partition_val=None):
    # Use the following method to overwrite to a delta table instead of "INSERT OVERWRITE" to avoid the error 'Table does not support dynamic overwrite in batch mode'
    if is_partitioned is False:
        df.write.format("delta").mode("overwrite").saveAsTable(table_name)
    else:
        df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("replaceWhere", f"{partition_col} = '{partition_val}'") \
        .partitionBy(partition_col) \
        .saveAsTable(table_name)

In [0]:
def combine_lpo_mmc_results(spark, run_date, db, table_names):
    df = spark.sql("""
                    SELECT vin, product_id, product_type, 'MMC' as product_category, score, dt 
                    FROM {db}.{mmc_output}
                    WHERE dt = '{run_date}'
                    UNION ALL
                    SELECT vin, product_id, product_type, 'LPO' as product_category, score, dt
                    FROM {db}.{lpo_output}
                    WHERE dt = '{run_date}'
                    ORDER BY vin, product_category, product_type, product_id
                    """.format(run_date = run_date,
                               db = db,
                               mmc_output = table_names['mmc_output'],
                               lpo_output = table_names['lpo_output']))
    dynamic_write_in_batch_mode(df, table_name=db+'.'+table_names['combined_output'], is_partitioned=True, partition_col='dt', partition_val=run_date)

In [0]:
combine_lpo_mmc_results(spark, run_date, db, table_names)